
### Task 9: Building a Hybrid Recommendation Model

### Objective

The objective of this task is to develop a Hybrid Recommendation Model by combining Content-Based Filtering and Collaborative Filtering.

The model uses both TED Talk content features and user preferences to generate personalized recommendations.

### Recommendation Techniques

1. Content-Based Filtering: Recommends TED Talks based on content similarity.
2. Collaborative Filtering: Recommends TED Talks based on the preferences of similar users.
3. Hybrid Recommendation: Combines the scores from both approaches using a weighted method.

### Dataset

The cleaned TED Talks dataset from previous tasks and the simulated user interaction dataset from Task 7 will be used.

Note: The user interaction data is simulated for educational and demonstration purposes.



### 1. Understanding Hybrid Recommendation

#### What is a Hybrid Recommendation System?

A Hybrid Recommendation System combines two or more recommendation techniques to generate personalized recommendations.

In this project, we combine Content-Based Filtering and Collaborative Filtering.

#### 1. Content-Based Filtering

Content-Based Filtering recommends TED Talks based on the similarity of their content, such as:

- Title
- Description
- Topics or tags
- Transcript

It identifies talks with content similar to a talk that the user has already shown interest in.

#### 2. Collaborative Filtering

Collaborative Filtering recommends TED Talks based on the preferences and ratings of users with similar interests.

It identifies similar users using Cosine Similarity and predicts ratings for talks that the target user has not rated.

#### 3. Hybrid Recommendation

Hybrid Recommendation combines the results of Content-Based Filtering and Collaborative Filtering.

It uses a weighted scoring approach to calculate a combined recommendation score.

#### Weighted Score Formula

Hybrid Score = (α × Content-Based Score) + (β × Collaborative Score)

Where:

- α (alpha) represents the weight assigned to Content-Based Filtering.
- β (beta) represents the weight assigned to Collaborative Filtering.
- α + β = 1.

For this project, we will initially use:

- Content-Based weight = 0.5
- Collaborative Filtering weight = 0.5

#### Advantages of Hybrid Recommendation

1. Combines content information and user preferences.
2. Provides more personalized recommendations.
3. Uses strengths of both recommendation approaches.
4. Can reduce some limitations of individual recommendation techniques.

#### Limitations

1. Requires both content and interaction data.
2. Weight selection can affect recommendation quality.
3. Sparse user interaction data may limit collaborative recommendations.
4. The quality of recommendations depends on the underlying models and datasets.


In [15]:

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

# Load cleaned TED Talks dataset
ted_df = pd.read_csv("../data/processed/ted_talks_cleaned.csv")

# Load simulated user interaction dataset
interactions_df = pd.read_csv("../data/interaction/user_interactions.csv")

# Display dataset information
print("TED Talks Dataset Shape:", ted_df.shape)
print("User Interactions Dataset Shape:", interactions_df.shape)

print("\nTED Talks Dataset:")
display(ted_df.head())

print("\nUser Interaction Dataset:")
display(interactions_df.head())


TED Talks Dataset Shape: (4005, 26)
User Interactions Dataset Shape: (37, 7)

TED Talks Dataset:


,talk_id,title,speaker_1,all_speakers,occupations,about_speakers,views,recorded_date,published_date,event,...,url,description,transcript,title_clean,description_clean,transcript_clean,topics_clean,combined_text,combined_text_length,combined_text_word_count
0,1,Averting the climate crisis,Al Gore,{0: 'Al Gore'},{0: ['climate advocate']},{0: 'Nobel Laureate Al Gore focused the world’...,3523392,2006-02-25,2006-06-27,TED2006,...,https://www.ted.com/talks/al_gore_averting_the...,With the same humor and humanity he exuded in ...,"Thank you so much, Chris. And it's truly a gre...",averting the climate crisis,with the same humor and humanity he exuded in ...,thank you so much chris and it s truly a great...,alternative energy cars climate change culture...,averting the climate crisis alternative energy...,11819,2269
1,92,The best stats you've ever seen,Hans Rosling,{0: 'Hans Rosling'},{0: ['global health expert; data visionary']},"{0: 'In Hans Rosling’s hands, data sings. Glob...",14501685,2006-02-22,2006-06-27,TED2006,...,https://www.ted.com/talks/hans_rosling_the_bes...,You've never seen data presented like this. Wi...,"About 10 years ago, I took on the task to teac...",the best stats you ve ever seen,you ve never seen data presented like this wit...,about 10 years ago i took on the task to teach...,africa asia google demo economics global issue...,the best stats you ve ever seen africa asia go...,17185,3278
2,7,Simplicity sells,David Pogue,{0: 'David Pogue'},{0: ['technology columnist']},{0: 'David Pogue is the personal technology co...,1920832,2006-02-24,2006-06-27,TED2006,...,https://www.ted.com/talks/david_pogue_simplici...,New York Times columnist David Pogue takes aim...,"(Music: ""The Sound of Silence,"" Simon & Garfun...",simplicity sells,new york times columnist david pogue takes aim...,music the sound of silence simon garfunkel hel...,computers entertainment interface design media...,simplicity sells computers entertainment inter...,17892,3548
3,53,Greening the ghetto,Majora Carter,{0: 'Majora Carter'},{0: ['activist for environmental justice']},{0: 'Majora Carter redefined the field of envi...,2664069,2006-02-26,2006-06-27,TED2006,...,https://www.ted.com/talks/majora_carter_greeni...,"In an emotionally charged talk, MacArthur-winn...",If you're here today — and I'm very happy that...,greening the ghetto,in an emotionally charged talk macarthur winni...,if you re here today and i m very happy that y...,macarthur grant activism business cities envir...,greening the ghetto macarthur grant activism b...,18150,3226
4,66,Do schools kill creativity?,Sir Ken Robinson,{0: 'Sir Ken Robinson'},"{0: ['author', 'educator']}","{0: ""Creativity expert Sir Ken Robinson challe...",65051954,2006-02-25,2006-06-27,TED2006,...,https://www.ted.com/talks/sir_ken_robinson_do_...,Sir Ken Robinson makes an entertaining and pro...,Good morning. How are you? (Audience) Good. It...,do schools kill creativity,sir ken robinson makes an entertaining and pro...,good morning how are you audience good it s be...,children creativity culture dance education pa...,do schools kill creativity children creativity...,17061,3326



User Interaction Dataset:


,user_id,talk_id,title,rating,liked,watched,bookmarked
0,U001,1,Averting the climate crisis,5,True,True,True
1,U001,92,The best stats you've ever seen,4,True,True,False
2,U001,7,Simplicity sells,3,False,True,False
3,U001,53,Greening the ghetto,5,True,True,True
4,U001,66,Do schools kill creativity?,2,False,True,False


In [16]:
#Build the Content-Based Recommendation Model

# Select the content features
content_df = ted_df[
    ["talk_id", "title", "title_clean", "description_clean", "topics_clean"]
].copy()

# Handle missing text values
content_features = [
    "title_clean",
    "description_clean",
    "topics_clean"
]

content_df[content_features] = content_df[content_features].fillna("")

# Combine selected text features
content_df["content_text"] = (
    content_df["title_clean"] + " " +
    content_df["description_clean"] + " " +
    content_df["topics_clean"]
)

# Remove duplicate TED Talk IDs
content_df = content_df.drop_duplicates(subset="talk_id").reset_index(drop=True)

# Convert text into TF-IDF vectors
tfidf = TfidfVectorizer(stop_words="english", max_features=5000)

content_vectors = tfidf.fit_transform(content_df["content_text"])

# Calculate content similarity
content_similarity = cosine_similarity(content_vectors)

# Create a similarity DataFrame using TED Talk IDs
content_similarity_df = pd.DataFrame(
    content_similarity,
    index=content_df["talk_id"],
    columns=content_df["talk_id"]
)

print("Content-Based Model Created Successfully!")
print("TF-IDF Matrix Shape:", content_vectors.shape)
print("Content Similarity Matrix Shape:", content_similarity_df.shape)


Content-Based Model Created Successfully!
TF-IDF Matrix Shape: (4005, 5000)
Content Similarity Matrix Shape: (4005, 4005)


In [17]:
#Build the Collaborative Filtering Model


# Create the User-Item Matrix
user_item_matrix = interactions_df.pivot_table(
    index="user_id",
    columns="talk_id",
    values="rating"
)

# Fill missing values for similarity calculation
user_item_filled = user_item_matrix.fillna(0)

# Calculate user similarity
user_similarity = cosine_similarity(user_item_filled)

# Create the User Similarity Matrix
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)

print("Collaborative Filtering Model Created Successfully!")

print("\nUser-Item Matrix:")
display(user_item_matrix)

print("\nUser Similarity Matrix:")
display(user_similarity_df.round(3))


Collaborative Filtering Model Created Successfully!

User-Item Matrix:


talk_id,1,7,49,53,66,86,92,94
user_id,,,,,,,,
U001,5.0,3.0,4.0,5.0,2.0,5.0,4.0,3.0
U002,4.0,2.0,3.0,4.0,5.0,4.0,5.0,NaN
U003,2.0,5.0,5.0,4.0,3.0,4.0,3.0,2.0
U004,5.0,4.0,4.0,2.0,5.0,NaN,4.0,NaN
U005,3.0,4.0,2.0,5.0,3.0,4.0,5.0,5.0



User Similarity Matrix:


user_id,U001,U002,U003,U004,U005
user_id,,,,,
U001,1.000,0.903,0.923,0.776,0.938
U002,0.903,1.000,0.868,0.874,0.852
U003,0.923,0.868,1.000,0.810,0.898
U004,0.776,0.874,0.810,1.000,0.732
U005,0.938,0.852,0.898,0.732,1.000


In [18]:
#Generate Content-Based Recommendations for a User


def get_content_recommendations(user_id, top_n=10):

    # Check whether the user exists
    if user_id not in user_item_matrix.index:
        print(f"User ID '{user_id}' was not found.")
        return pd.DataFrame(columns=["talk_id", "content_score"])

    # Find talks rated 4 or 5 by the user
    liked_talks = user_item_matrix.loc[user_id]
    liked_talks = liked_talks[liked_talks >= 4].index

    if len(liked_talks) == 0:
        print("No highly rated talks found for this user.")
        return pd.DataFrame(columns=["talk_id", "content_score"])

    # Find talks already rated by the user
    rated_talks = user_item_matrix.loc[user_id].dropna().index

    # Calculate the maximum similarity to the user's liked talks
    similarity_scores = content_similarity_df.loc[:, liked_talks].max(axis=1)

    # Exclude talks already rated
    similarity_scores = similarity_scores.drop(
        labels=rated_talks, errors="ignore"
    )

    # Sort and return the top recommendations
    recommendations = similarity_scores.sort_values(
        ascending=False
    ).head(top_n)

    return recommendations.rename("content_score").reset_index().rename(
    columns={"index": "talk_id"}
)


# Test content-based recommendations
content_results = get_content_recommendations("U002", top_n=10)

print("Content-Based Recommendations for U002:")
display(content_results)


Content-Based Recommendations for U002:


,talk_id,content_score
0,865,0.436199
1,243,0.391374
2,620,0.380255
3,535,0.373878
4,2441,0.342382
5,140,0.337244
6,51,0.331965
7,1380,0.314429
8,2093,0.308977
9,30297,0.306289


In [19]:
#Generate Collaborative Filtering Recommendations


def get_collaborative_recommendations(user_id, top_n=10):

    # Check whether the user exists
    if user_id not in user_item_matrix.index:
        print(f"User ID '{user_id}' was not found.")
        return pd.DataFrame(columns=["talk_id", "collaborative_score"])

    # Find talks not yet rated by the target user
    user_ratings = user_item_matrix.loc[user_id]
    unrated_talks = user_ratings[user_ratings.isna()].index

    # Find similar users
    similar_users = (
        user_similarity_df[user_id]
        .drop(user_id)
        .sort_values(ascending=False)
    )

    recommendations = []

    # Predict ratings for unrated talks
    for talk_id in unrated_talks:

        ratings = user_item_matrix[talk_id].dropna()

        # Find users who rated this talk
        common_users = similar_users.index.intersection(ratings.index)

        if len(common_users) == 0:
            continue

        weights = similar_users.loc[common_users]
        user_ratings_for_talk = ratings.loc[common_users]

        if weights.sum() <= 0:
            continue

        predicted_rating = (
            weights * user_ratings_for_talk
        ).sum() / weights.sum()

        recommendations.append({
            "talk_id": talk_id,
            "collaborative_score": predicted_rating
        })

    recommendations_df = pd.DataFrame(recommendations)

    if recommendations_df.empty:
        return pd.DataFrame(
            columns=["talk_id", "collaborative_score"]
        )

    return (
        recommendations_df
        .sort_values("collaborative_score", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )


# Test collaborative filtering recommendations
collaborative_results = get_collaborative_recommendations(
    "U002", top_n=10
)

print("Collaborative Filtering Recommendations for U002:")
display(collaborative_results)


Collaborative Filtering Recommendations for U002:


,talk_id,collaborative_score
0,94,3.319202


In [20]:
#Combine Both Recommendation Results
#(combining Content-Based Filtering and Collaborative Filtering.)


# Define weights for both recommendation models
content_weight = 0.5
collaborative_weight = 0.5

# Copy recommendation results
content_scores = content_results.copy()
collaborative_scores = collaborative_results.copy()

# Normalize collaborative ratings from 1-5 to 0-1
collaborative_scores["collaborative_score"] = (
    collaborative_scores["collaborative_score"] - 1
) / 4

# Combine both recommendation lists using an outer join
hybrid_df = pd.merge(
    content_scores,
    collaborative_scores,
    on="talk_id",
    how="outer"
)

# Fill missing scores with zero
hybrid_df[["content_score", "collaborative_score"]] = (
    hybrid_df[["content_score", "collaborative_score"]].fillna(0)
)

# Calculate the weighted hybrid score
hybrid_df["hybrid_score"] = (
    content_weight * hybrid_df["content_score"] +
    collaborative_weight * hybrid_df["collaborative_score"]
)

# Sort recommendations by hybrid score
hybrid_df = hybrid_df.sort_values(
    "hybrid_score",
    ascending=False
).reset_index(drop=True)

print("Combined Hybrid Recommendation Results:")
display(hybrid_df.head(10).round(3))



Combined Hybrid Recommendation Results:


,talk_id,content_score,collaborative_score,hybrid_score
0,94,0.000,0.58,0.290
1,865,0.436,0.00,0.218
2,243,0.391,0.00,0.196
3,620,0.380,0.00,0.190
4,535,0.374,0.00,0.187
5,2441,0.342,0.00,0.171
6,140,0.337,0.00,0.169
7,51,0.332,0.00,0.166
8,1380,0.314,0.00,0.157
9,2093,0.309,0.00,0.154


In [21]:
#Build the Reusable Hybrid Recommendation Function


def hybrid_recommend(user_id, top_n=5,
                     content_weight=0.5,
                     collaborative_weight=0.5):

    # Validate user ID
    if user_id not in user_item_matrix.index:
        print(f"User ID '{user_id}' was not found.")
        return pd.DataFrame(
            columns=["talk_id", "content_score",
                     "collaborative_score", "hybrid_score"]
        )

    # Validate top_n
    if not isinstance(top_n, int) or isinstance(top_n, bool) or top_n <= 0:
        print("top_n must be a positive integer.")
        return pd.DataFrame(
            columns=["talk_id", "content_score",
                     "collaborative_score", "hybrid_score"]
        )

    # Validate weights
    if (content_weight < 0 or collaborative_weight < 0 or
        not np.isclose(content_weight + collaborative_weight, 1)):
        print("Weights must be non-negative and add up to 1.")
        return pd.DataFrame(
            columns=["talk_id", "content_score",
                     "collaborative_score", "hybrid_score"]
        )

    # Generate recommendations from both models
    content_results = get_content_recommendations(user_id, top_n=top_n)
    collaborative_results = get_collaborative_recommendations(
        user_id, top_n=top_n
    )

    # Combine both recommendation lists
    hybrid_df = pd.merge(
        content_results,
        collaborative_results,
        on="talk_id",
        how="outer"
    )

    if hybrid_df.empty:
        print("No recommendations available for this user.")
        return pd.DataFrame(
            columns=["talk_id", "content_score",
                     "collaborative_score", "hybrid_score"]
        )

    # Normalize collaborative ratings from 1-5 to 0-1
    hybrid_df["collaborative_score"] = (
        hybrid_df["collaborative_score"].fillna(1) - 1
    ) / 4

    # Fill missing content scores
    hybrid_df["content_score"] = hybrid_df["content_score"].fillna(0)

    # Calculate weighted hybrid score
    hybrid_df["hybrid_score"] = (
        content_weight * hybrid_df["content_score"] +
        collaborative_weight * hybrid_df["collaborative_score"]
    )

    # Remove duplicate TED Talk IDs and rank recommendations
    hybrid_df = (
        hybrid_df.drop_duplicates(subset="talk_id")
        .sort_values("hybrid_score", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )

    return hybrid_df


print("Hybrid Recommendation Function Created Successfully!")


Hybrid Recommendation Function Created Successfully!


In [22]:
# Test the Hybrid Recommendation Model with different users

test_users = ["U002", "U004", "U001"]

for user in test_users:

    print(f"\nHybrid Recommendations for {user}:")

    hybrid_results = hybrid_recommend(user, top_n=5)

    if hybrid_results.empty:
        print("No recommendations available.")
    else:
        display(hybrid_results.round(3))



Hybrid Recommendations for U002:


,talk_id,content_score,collaborative_score,hybrid_score
0,94,0.000,0.58,0.290
1,865,0.436,0.00,0.218
2,243,0.391,0.00,0.196
3,620,0.380,0.00,0.190
4,535,0.374,0.00,0.187



Hybrid Recommendations for U004:


,talk_id,content_score,collaborative_score,hybrid_score
0,86,0.000,0.811,0.405
1,94,0.000,0.571,0.285
2,865,0.436,0.000,0.218
3,243,0.391,0.000,0.196
4,845,0.383,0.000,0.192



Hybrid Recommendations for U001:


,talk_id,content_score,collaborative_score,hybrid_score
0,243,0.391,0.0,0.195687
1,620,0.380,0.0,0.190127
2,535,0.374,0.0,0.186939
3,418,0.358,0.0,0.178986
4,2441,0.342,0.0,0.171191


In [23]:
#Display TED Talk titles

# Create a lookup table for TED Talk IDs and titles
talk_lookup = ted_df[["talk_id", "title"]].drop_duplicates(
    subset="talk_id"
)

# Display hybrid recommendations with TED Talk titles
for user in ["U002", "U004", "U001"]:
    print(f"\nHybrid Recommendations for {user}:")

    recommendations = hybrid_recommend(user, top_n=5)

    if not recommendations.empty:
        final_recommendations = recommendations.merge(
            talk_lookup,
            on="talk_id",
            how="left"
        )

        final_recommendations = final_recommendations[
            [
                "talk_id",
                "title",
                "content_score",
                "collaborative_score",
                "hybrid_score"
            ]
        ]

        display(final_recommendations.round(3))
        


Hybrid Recommendations for U002:


,talk_id,title,content_score,collaborative_score,hybrid_score
0,94,Let's teach religion -- all religion -- in sch...,0.000,0.58,0.290
1,865,Bring on the learning revolution!,0.436,0.00,0.218
2,243,New thinking on the climate crisis,0.391,0.00,0.196
3,620,Let my dataset change your mindset,0.380,0.00,0.190
4,535,What comes after An Inconvenient Truth?,0.374,0.00,0.187



Hybrid Recommendations for U004:


,talk_id,title,content_score,collaborative_score,hybrid_score
0,86,Letting go of God,0.000,0.811,0.405
1,94,Let's teach religion -- all religion -- in sch...,0.000,0.571,0.285
2,865,Bring on the learning revolution!,0.436,0.000,0.218
3,243,New thinking on the climate crisis,0.391,0.000,0.196
4,845,Toward a science of simplicity,0.383,0.000,0.192



Hybrid Recommendations for U001:


,talk_id,title,content_score,collaborative_score,hybrid_score
0,243,New thinking on the climate crisis,0.391,0.0,0.195687
1,620,Let my dataset change your mindset,0.380,0.0,0.190127
2,535,What comes after An Inconvenient Truth?,0.374,0.0,0.186939
3,418,My library of human imagination,0.358,0.0,0.178986
4,2441,The case for optimism on climate change,0.342,0.0,0.171191


#### Comparison of Recommendation Models

The recommendations generated by Content-Based Filtering, Collaborative Filtering, and the Hybrid Model are compared for the same user.

This comparison helps evaluate how combining content similarity and user preferences affects the final recommendation results.

In [24]:
# Select a user for comparison
user_id = "U002"

# Generate recommendations from all three models
content_results = get_content_recommendations(user_id, top_n=5)
collaborative_results = get_collaborative_recommendations(user_id, top_n=5)
hybrid_results = hybrid_recommend(user_id, top_n=5)

# Add TED Talk titles
content_comparison = content_results.merge(
    talk_lookup, on="talk_id", how="left"
)
collaborative_comparison = collaborative_results.merge(
    talk_lookup, on="talk_id", how="left"
)
hybrid_comparison = hybrid_results.merge(
    talk_lookup, on="talk_id", how="left"
)

# Display individual model results
print("CONTENT-BASED RECOMMENDATIONS")
display(content_comparison)

print("COLLABORATIVE FILTERING RECOMMENDATIONS")
display(collaborative_comparison)

print("HYBRID RECOMMENDATIONS")
display(hybrid_comparison)

CONTENT-BASED RECOMMENDATIONS


,talk_id,content_score,title
0,865,0.436199,Bring on the learning revolution!
1,243,0.391374,New thinking on the climate crisis
2,620,0.380255,Let my dataset change your mindset
3,535,0.373878,What comes after An Inconvenient Truth?
4,2441,0.342382,The case for optimism on climate change


COLLABORATIVE FILTERING RECOMMENDATIONS


,talk_id,collaborative_score,title
0,94,3.319202,Let's teach religion -- all religion -- in sch...


HYBRID RECOMMENDATIONS


,talk_id,content_score,collaborative_score,hybrid_score,title
0,94,0.000000,0.5798,0.289900,Let's teach religion -- all religion -- in sch...
1,865,0.436199,0.0000,0.218100,Bring on the learning revolution!
2,243,0.391374,0.0000,0.195687,New thinking on the climate crisis
3,620,0.380255,0.0000,0.190127,Let my dataset change your mindset
4,535,0.373878,0.0000,0.186939,What comes after An Inconvenient Truth?


#### Observations and Analysis

1. **Content-Based Filtering:** Recommended five TED Talks based on their content similarity to the talks liked by user U002. The highest-ranked recommendation was *Bring on the Learning Revolution!* with a similarity score of approximately 0.436.

2. **Collaborative Filtering:** Recommended one TED Talk, *Let's Teach Religion — All Religion — in Schools!* with a predicted rating of approximately 3.32 out of 5.

3. **Hybrid Recommendation:** Combined the content-based and collaborative filtering scores using equal weights (0.5 each).

4. The Hybrid Model ranked *Let's Teach Religion — All Religion — in Schools!* first because it received a strong collaborative filtering score.

5. The remaining recommendations came from Content-Based Filtering because no collaborative scores were available for those talks.

6. The Hybrid Model successfully combined recommendations from both approaches and produced a single ranked recommendation list.

7. Collaborative Filtering provided limited recommendations because the simulated user interaction dataset was small and sparse.

#### Conclusion

The Hybrid Recommendation Model successfully combines Content-Based Filtering and User-Based Collaborative Filtering to generate personalized TED Talk recommendations. It can provide recommendations even when collaborative interaction data is limited by using content similarity. Therefore, the hybrid approach offers a more flexible recommendation strategy than relying on either technique alone.

In [25]:
#Test invalid inputs

# Test invalid user
print("Invalid User Test:")
display(hybrid_recommend("U999", top_n=5))

# Test invalid top_n
print("\nInvalid top_n Test:")
display(hybrid_recommend("U002", top_n=0))

# Test invalid weights
print("\nInvalid Weights Test:")
display(hybrid_recommend(
    "U002",
    top_n=5,
    content_weight=0.8,
    collaborative_weight=0.5
))

Invalid User Test:
User ID 'U999' was not found.


,talk_id,content_score,collaborative_score,hybrid_score



Invalid top_n Test:
top_n must be a positive integer.


,talk_id,content_score,collaborative_score,hybrid_score



Invalid Weights Test:
Weights must be non-negative and add up to 1.


,talk_id,content_score,collaborative_score,hybrid_score


In [26]:
#Creating the Task 9 observations PDF

from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
import os

# Create reports folder if it does not exist
os.makedirs("../reports", exist_ok=True)

# Set PDF file path
pdf_path = "../reports/task_09_observations.pdf"

# Create PDF document
doc = SimpleDocTemplate(
    pdf_path,
    pagesize=A4,
    rightMargin=50,
    leftMargin=50,
    topMargin=50,
    bottomMargin=50
)

styles = getSampleStyleSheet()
story = []

# Title
story.append(Paragraph(
    "Task 9: Building a Hybrid Recommendation Model",
    styles["Title"]
))
story.append(Spacer(1, 15))

# Report content
report_sections = [
    ("1. Objective",
     "To develop a Hybrid Recommendation Model by combining "
     "Content-Based Filtering and User-Based Collaborative Filtering "
     "to generate personalized TED Talk recommendations."),

    ("2. Dataset",
     "The cleaned TED Talks dataset and the simulated user interaction "
     "dataset created in Task 7 were used. The interaction dataset contains "
     "user IDs, TED Talk IDs, ratings, and interaction details."),

    ("3. Methodology",
     "The Content-Based Filtering model uses TF-IDF and Cosine Similarity "
     "to identify similar TED Talks based on their titles, descriptions, "
     "and topics. The Collaborative Filtering model uses user ratings "
     "and Cosine Similarity to identify recommendations based on similar users."),

    ("4. Hybrid Recommendation Model",
     "A reusable hybrid_recommend() function was developed to combine "
     "recommendations from both models. Collaborative ratings were normalized "
     "from a 1-5 scale to a 0-1 scale. The final hybrid score was calculated "
     "using equal weights of 0.5 for Content-Based and Collaborative scores. "
     "Duplicate recommendations were removed and the results were ranked "
     "according to their hybrid scores."),

    ("5. Testing and Results",
     "The hybrid model was tested with users U002, U004, and U001. "
     "For U002, the model combined one collaborative recommendation with "
     "content-based recommendations. For U004, it combined two collaborative "
     "recommendations with content-based recommendations. For U001, "
     "content-based recommendations were generated because no collaborative "
     "recommendations were available."),

    ("6. Observations",
     "The hybrid model successfully combined both recommendation techniques. "
     "Collaborative Filtering contributed recommendations based on user ratings, "
     "while Content-Based Filtering provided recommendations based on talk "
     "similarity. When collaborative recommendations were unavailable, "
     "the model continued to generate recommendations using content similarity."),

    ("7. Validation and Limitations",
     "The model was tested with invalid user IDs, invalid recommendation "
     "counts, and invalid weights. All validation tests worked correctly. "
     "The system is limited by the small and simulated user interaction "
     "dataset, which may result in sparse collaborative recommendations."),

    ("8. Conclusion",
     "The Hybrid Recommendation Model was successfully developed by combining "
     "Content-Based Filtering and User-Based Collaborative Filtering. "
     "It provides a flexible recommendation approach by using both talk "
     "content and user preferences, especially when interaction data is limited. "
     "The user interaction data used in this project was simulated for "
     "educational and demonstration purposes.")
]

for heading, content in report_sections:
    story.append(Paragraph(heading, styles["Heading2"]))
    story.append(Spacer(1, 6))
    story.append(Paragraph(content, styles["BodyText"]))
    story.append(Spacer(1, 12))

# Generate PDF
doc.build(story)

print("PDF report created successfully!")
print("Saved at:", pdf_path)


PDF report created successfully!
Saved at: ../reports/task_09_observations.pdf
